In [1]:
# ============================================================
# 05_ffs_assessment.ipynb  —  API 579-1/ASME FFS-1  Level 1
# Evaluación Fitness-for-Service — Flota completa Mongstad
# ============================================================
import sys
sys.path.append('../')
from src.ffs_engine import FFSEngine

ENGINE = FFSEngine(
    static_data_path=r'C:\Users\User\Desktop\TWIN-RBI-MONGSTAD\data\raw\pipe_static_data.csv',
    rbi_labels_path=r'C:\Users\User\Desktop\TWIN-RBI-MONGSTAD\data\processed\rbi_labels.csv'
)
print('✅ Engine listo')

✅ Engine listo


In [2]:
TAG = 'V-101'   # ← cambiar aquí para evaluar otro activo

r = ENGINE.assess(TAG)
print(f"{'='*45}")
print(f"  FFS Assessment — {r.tag}  [{r.asset_type}]")
print(f"{'='*45}")
print(f"  t_nominal        : {r.t_nominal_mm} mm")
print(f"  t_actual         : {r.t_actual_mm} mm")
print(f"  t_min            : {r.t_min_mm} mm")
print(f"  t_margin         : {r.t_margin_mm} mm")
print(f"  MAWP             : {r.MAWP_bar} bar  |  P_oper: {r.P_oper_bar} bar")
print(f"{'─'*45}")
print(f"  GML              : {r.gml_status}")
print(f"  LML / Pitting    : {r.lml_status}")
print(f"  SSC / HIC        : {r.ssc_status}")
print(f"  Creep            : {r.creep_status}")
print(f"  Fatiga térmica   : {r.fatigue_status}")
print(f"{'─'*45}")
print(f"  FFS Status       : {r.ffs_status}")
print(f"  Vida remanente   : {r.remaining_life_yr} años")
print(f"  Acción           : {r.action_required}")
print(f"  Próx. inspección : {r.next_insp_months} meses")

  FFS Assessment — V-101  [vessel]
  t_nominal        : 22.0 mm
  t_actual         : 13.086 mm
  t_min            : 8.5 mm
  t_margin         : 3.903 mm
  MAWP             : 21.93 bar  |  P_oper: 16.5 bar
─────────────────────────────────────────────
  GML              : PASS
  LML / Pitting    : PASS
  SSC / HIC        : FAIL — SSC activo (NACE MR0175)
  Creep            : PASS
  Fatiga térmica   : PASS
─────────────────────────────────────────────
  FFS Status       : FAIL
  Vida remanente   : 11.0 años
  Acción           : Acción inmediata requerida — revisar con ingeniero certificado API 579
  Próx. inspección : 3 meses


In [3]:
df_pfi = ENGINE.pfi_all()
print(df_pfi[['color','tag','PFI_final','PFI_thickness',
              'PFI_mawp','PFI_rul','dominant',
              'level','RUL_years']].to_string(index=False))

color   tag  PFI_final  PFI_thickness  PFI_mawp  PFI_rul            dominant   level  RUL_years
    🟠  L-10       99.9           99.6      13.6     99.9 Vida útil consumida CRÍTICO       0.02
    🟠  L-02       99.7           99.5      18.1     99.7 Vida útil consumida CRÍTICO       0.05
    🟠  L-03       99.6           99.3      59.5     99.6 Vida útil consumida CRÍTICO       0.07
    🟠  L-01       98.4           97.9      49.0     98.4 Vida útil consumida CRÍTICO       0.31
    🟠  L-05       84.2           83.2      16.2     84.2 Vida útil consumida CRÍTICO       3.56
    🟠  L-04       75.4           75.4      59.4     72.3  Pérdida de espesor CRÍTICO       6.13
    🟠 V-101       75.2           66.0      75.2     63.4     Presión vs MAWP CRÍTICO      10.96
    🟡  L-06       73.8           73.8       7.2     73.5  Pérdida de espesor MONITOR       5.05
    🟡 E-101       73.8           73.8      65.9     73.2  Pérdida de espesor MONITOR       6.95
    🟡  L-08       52.2           52.2   

In [4]:
df_all = ENGINE.assess_all()
print(df_all[['tag','ffs_status','t_actual_mm','t_margin_mm',
              'remaining_life_yr','action_required',
              'next_insp_months']].to_string(index=False))

  tag ffs_status  t_actual_mm  t_margin_mm  remaining_life_yr                                                        action_required  next_insp_months
 L-01    MONITOR        4.088        0.088                0.3               Inspección aumentada — programar NDT en próximos 6 meses                 6
 L-02    MONITOR        2.814        0.014                0.0               Inspección aumentada — programar NDT en próximos 6 meses                 6
 L-03    MONITOR        3.524        0.024                0.0               Inspección aumentada — programar NDT en próximos 6 meses                 6
 L-04    MONITOR        3.742        0.742                6.1               Inspección aumentada — programar NDT en próximos 6 meses                 6
 L-05    MONITOR        4.108        0.608                3.8               Inspección aumentada — programar NDT en próximos 6 meses                 6
 L-06    MONITOR        3.505        0.705                5.1               Inspección aumenta

In [5]:
import pandas as pd
df_static = pd.read_csv(r'C:\Users\User\Desktop\TWIN-RBI-MONGSTAD\data\raw\pipe_static_data.csv')
print(df_static[df_static['line_id'].isin(['L-10','L-03'])][
    ['line_id','nominal_thickness_mm','tmin_mm',
     'last_inspection_year','install_year']])

df_rbi = pd.read_csv(r'C:\Users\User\Desktop\TWIN-RBI-MONGSTAD\data\processed\rbi_labels.csv')
print(df_rbi[df_rbi['line_id'].isin(['L-10','L-03'])][
    ['line_id','cr_mean','t_estimated','RUL_years']].tail(4))

  line_id  nominal_thickness_mm  tmin_mm  last_inspection_year  install_year
2    L-03                  7.11      3.5                  2015          2005
9    L-10                  5.49      2.8                  2014          2005
    line_id  cr_mean  t_estimated  RUL_years
205    L-10  0.45160        2.810       0.02
206    L-10  0.45160        2.810       0.02
207    L-10  0.70542        2.843       0.10
208    L-10  0.70578        2.810       0.02
